# 03 — Gold Marts (Star Schema)

Builds `dim_locatie`, `dim_tijd`, `fact_verkeer`, `fact_waterpeil`, `fact_waterkwaliteit`, and the cross-domain alert mart from the Silver layer.

**Prerequisite:** run `02_silver_curate` first.

In [ ]:
import pandas as pd

def read_silver(name):
    return spark.read.format('delta').table(f'silver_{name.lower()}').toPandas()

def write_gold(name, df):
    spark.createDataFrame(df).write.mode('overwrite').format('delta').saveAsTable(f'gold_{name.lower()}')
    print(f'  gold_{name}: {len(df):>7} rows')

In [ ]:
wegvak    = read_silver('wegvak')
watergang = read_silver('watergang')
verkeer   = read_silver('verkeersmeting')
peil      = read_silver('waterpeilmeting')
kwaliteit = read_silver('waterkwaliteitsmeting')

# dim_locatie: unifies road segments and waterways
dim_locatie = pd.concat([
    wegvak.assign(locatie_type='wegvak').rename(columns={'wegvak_id': 'locatie_id', 'straatnaam': 'naam'})
          [['locatie_id', 'naam', 'locatie_type', 'wijk_code', 'geom_x_rd', 'geom_y_rd']],
    watergang.assign(locatie_type='watergang', geom_x_rd=None, geom_y_rd=None)
             .rename(columns={'watergang_id': 'locatie_id'})
             [['locatie_id', 'naam', 'locatie_type', 'wijk_code', 'geom_x_rd', 'geom_y_rd']],
], ignore_index=True)
write_gold('dim_locatie', dim_locatie)

# dim_tijd: hourly time dimension
times = pd.concat([
    pd.to_datetime(verkeer['meet_tijdstip']),
    pd.to_datetime(peil['meet_tijdstip']),
    pd.to_datetime(kwaliteit['meet_tijdstip']),
]).drop_duplicates().sort_values()
dim_tijd = pd.DataFrame({'tijdstip': times})
dim_tijd['datum']      = dim_tijd['tijdstip'].dt.date
dim_tijd['uur']        = dim_tijd['tijdstip'].dt.hour
dim_tijd['weekdag']    = dim_tijd['tijdstip'].dt.day_name()
dim_tijd['is_weekend'] = dim_tijd['tijdstip'].dt.weekday >= 5
write_gold('dim_tijd', dim_tijd)

In [ ]:
# Fact tables
write_gold('fact_verkeer',        verkeer.rename(columns={'wegvak_id': 'locatie_id'}))
write_gold('fact_waterpeil',      peil.rename(columns={'watergang_id': 'locatie_id'}))
write_gold('fact_waterkwaliteit', kwaliteit.rename(columns={'watergang_id': 'locatie_id'}))

In [ ]:
# Cross-domain alert mart: hours where traffic peaks AND water norm breaches co-occur per wijk
overschrijdingen = kwaliteit[kwaliteit['norm_status'] == 'overschrijding'].copy()
overschrijdingen['uur'] = pd.to_datetime(overschrijdingen['meet_tijdstip']).dt.floor('h')
overschrijdingen = overschrijdingen.merge(watergang[['watergang_id', 'wijk_code']], on='watergang_id')

verkeer_h = verkeer.copy()
verkeer_h['uur'] = pd.to_datetime(verkeer_h['meet_tijdstip']).dt.floor('h')
verkeer_h = verkeer_h.merge(wegvak[['wegvak_id', 'wijk_code']], on='wegvak_id')

cross = (
    verkeer_h.groupby(['wijk_code', 'uur'], as_index=False)['voertuigen_per_uur'].sum()
    .merge(
        overschrijdingen.groupby(['wijk_code', 'uur'], as_index=False).size()
                        .rename(columns={'size': 'overschrijdingen'}),
        on=['wijk_code', 'uur'], how='inner',
    )
)
write_gold('fact_cross_domain_alert', cross)

print('\n✅ Gold layer complete!')